In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/DeepLoc_Embeddings/'

import os
os.makedirs(SAVE_PATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
csv_path = '/content/drive/MyDrive/Swissprot_Membrane_Train_Validation_dataset.csv'
df = pd.read_csv(csv_path)
test_df = df.head(100)

In [ ]:
from transformers import pipeline
pipe = pipeline("fill-mask", model="facebook/esm2_t33_650M_UR50D")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

GPU 사용 가능: True
디바이스: Tesla T4


In [ ]:
from transformers import EsmTokenizer, EsmModel

model_name = "facebook/esm2_t33_650M_UR50D"  # 650M 풀파워!
tokenizer = EsmTokenizer.from_pretrained(model_name)
model = EsmModel.from_pretrained(model_name).to(device)
model.eval()

print(f"✅ 650M ESM-2 로드: {type(model)}")
print(f"차원: {model.config.hidden_size}")  # 1280 확인


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ 650M ESM-2 로드: <class 'transformers.models.esm.modeling_esm.EsmModel'>
차원: 1280


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft

class DCTAttentionPooling(nn.Module):
    """
    DeepLoc 2.1의 Attention + DCT Regularization 완전 구현
    """
    def __init__(self, embed_dim=320, num_heads=8):  # esm2_t6_8M은 320차원
        super().__init__()
        self.embed_dim = embed_dim
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.dct_scale = nn.Parameter(torch.tensor(0.1))  # 학습 가능한 DCT 강도

    def dct_prior(self, attn_weights):
        """
        Discrete Cosine Transform regularization
        Low-frequency 강조 → interpretable attention
        """
        # attn_weights: (batch, seq_len, seq_len)
        seq_len = attn_weights.shape[1]

        # 1D DCT (간단 버전)
        attn_mean = attn_weights.mean(dim=-1)  # (batch, seq_len)
        dct = torch.fft.rfft(attn_mean, dim=1).real  # 주파수 도메인

        # Low-frequency만 유지 (고주파 억제)
        dct[:, 5:] *= 0.3  # 5개 이후 고주파 감쇠
        dct_reg = torch.fft.irfft(dct, n=seq_len)

        # 정규화
        dct_reg = F.softmax(dct_reg, dim=-1)
        return dct_reg

    def forward(self, x):
        """
        x: (batch, seq_len, embed_dim)
        """
        batch_size, seq_len, _ = x.shape

        # Self-attention
        attn_out, attn_weights = self.attention(x, x, x)

        # DCT regularization
        dct_weights = self.dct_prior(attn_weights)  # (batch, seq_len)

        # Attention-weighted pooling
        pooled = torch.sum(attn_out * dct_weights.unsqueeze(-1), dim=1)

        return pooled, attn_weights, dct_weights  # (batch, embed_dim)


In [ ]:
import numpy as np
import torch
from tqdm import tqdm

def extract_deeploc_embeddings(sequences, model, tokenizer, pooler, device, batch_size=4):
    """
    ESM-2 + DCT Attention Pooling (DeepLoc 완전 구현)
    """
    embeddings_list = []

    for i in tqdm(range(0, len(sequences), batch_size), desc="DeepLoc 임베딩"):
        batch_seqs = sequences[i:i+batch_size]

        # ESM-2로 per-position embeddings
        inputs = tokenizer(batch_seqs, return_tensors="pt",
                          padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)  # ESM-2 모델!
            last_hidden = outputs.last_hidden_state  # (batch, seq_len, 320)

            # DCT Attention Pooling 적용!
            pooled, attn_weights, dct_weights = pooler(last_hidden)  # DeepLoc 마법!

        embeddings_list.append(pooled.cpu().numpy())
        torch.cuda.empty_cache()

    return np.vstack(embeddings_list)

In [ ]:
import torch
from torch.utils.data import dataset, DataLoader

class ProteinDataset(Dataset):
    """
    임베딩과 라벨을 담는 Dataset 클래스
    """
    def __init__(self, embeddings, labels):
        self.embeddings = torch.FloatTensor(embeddings).float()
        self.labels = torch.FloatTensor(labels).float()

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]

# DataLoader 생성
train_loader = DataLoader(dataset, batch_size=128, shuffle=True)

print(f"✅ DataLoader 생성 완료")
print(f"배치 크기: 128")
print(f"배치 수: {len(train_loader)}")

# 첫 배치 확인
batch_embeddings, batch_labels = next(iter(train_loader))
print(f"\n첫 배치:")
print(f"  임베딩: {batch_embeddings.shape}")
print(f"  라벨: {batch_labels.shape}")


TypeError: object of type 'module' has no len()

In [ ]:
from torch.utils.data import DataLoader, Dataset
from torchvision.ops import sigmoid_focal_loss
import numpy as np


focal_loss_fn = sigmoid_focal_loss

# DeepLoc 임베딩 생성
# 1. Pooler 생성
pooler = DCTAttentionPooling(embed_dim=1280, num_heads=8).to(device).eval()

# 2. 임베딩 생성
sequences = test_df['Sequence'].tolist()
embeddings_deeploc = extract_deeploc_embeddings(
    test_df['Sequence'].tolist(),
    model,
    tokenizer,
    pooler,
    device,
    batch_size=8  # 최적!
)

In [ ]:
# Partition 정보로 분할 & 저장
partitions = test_df['Partition'].values
partition_data = {}
for p in range(5):
    mask = partitions == p
    partition_data[p] = {
        'embeddings': embeddings_deeploc[mask],
        'labels': test_df.loc[mask, ['Peripheral','Transmembrane','LipidAnchor','Soluble']].values
    }
    np.save(f'{SAVE_PATH}deeploc_p{p}_embeddings.npy', partition_data[p]['embeddings'])
    np.save(f'{SAVE_PATH}deeploc_p{p}_labels.npy', partition_data[p]['labels'])

In [ ]:
embeddings = np.vstack([np.load(f'{SAVE_PATH}deeploc_p{i}_embeddings.npy') for i in range(5)])
labels = np.vstack([np.load(f'{SAVE_PATH}deeploc_p{i}_labels.npy') for i in range(5)])


In [ ]:
class ESM2DeepLoc(nn.Module):
    def __init__(self, esm_dim=1280):
        super().__init__()

        # Fusion 대신 ESM-2 projection
        self.esm_proj = nn.Sequential(
            nn.Linear(esm_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        # Multi-scale branches (DeepLoc 따라하기)
        self.branch1 = nn.Sequential(  # Large scale
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2)
        )
        self.branch2 = nn.Sequential(  # Medium scale
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2)
        )
        self.branch3 = nn.Linear(512, 64)  # Small scale

        # Final classifier
        self.classifier = nn.Linear(256 + 64 + 64, 4)

    def forward(self, x):  # x: (batch, 1280)
        x = self.esm_proj(x)

        b1 = self.branch1(x)      # (batch, 256)
        b2 = self.branch2(x)      # (batch, 64)
        b3 = self.branch3(x)      # (batch, 64)

        combined = torch.cat([b1, b2, b3], dim=1)  # (batch, 384)
        out = self.classifier(combined)
        return torch.sigmoid(out)


In [ ]:
import torch.nn.functional as F
from torch.nn import BCEWithLogitsLoss

def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for batch_embeddings, batch_labels in train_loader:
        batch_embeddings = batch_embeddings.to(device)
        batch_labels = batch_labels.to(device)

        optimizer.zero_grad()
        outputs = model(batch_embeddings)
        loss = criterion(outputs, batch_labels)  # scalar!
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch_embeddings, batch_labels in val_loader:
            batch_embeddings, batch_labels = batch_embeddings.to(device), batch_labels.to(device)
            outputs = model(batch_embeddings)
            loss = criterion(outputs, batch_labels)  # scalar!
            total_loss += loss.item()
    return total_loss / len(val_loader)

In [ ]:
import torch.nn as nn
import numpy as np
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch

# 0. partitions 정의 (주석 해제 또는 생성)
partitions = np.concatenate([np.full(len(np.load(f'{SAVE_PATH}deeploc_p{p}_embeddings.npy')), p) for p in range(5)])

# 1. 데이터 (CPU numpy 유지)
embeddings_deeploc = embeddings.cpu().numpy()  # GPU → CPU numpy!
labels_deeploc = labels.cpu().numpy()
print(f"DeepLoc 데이터: {embeddings_deeploc.shape}")


criterion = nn.BCEWithLogitsLoss()
# 2. 4-Fold 루프
fold_results = []
for fold in range(4):
    print(f"\n=== Fold {fold} ===")

    val_mask = (partitions == fold)
    train_mask = (partitions != fold) & (partitions != 4)

    # Numpy 배열로 슬라이싱 (CPU에서!)
    train_emb = embeddings_deeploc[train_mask]
    train_lab = labels_deeploc[train_mask]
    val_emb = embeddings_deeploc[val_mask]
    val_lab = labels_deeploc[val_mask]

    print(f"  Train: {train_emb.shape}, Val: {val_emb.shape}")

    # ✅ 수정1: numpy 직접 전달
    train_dataset = ProteinDataset(train_emb, train_lab)
    val_dataset = ProteinDataset(val_emb, val_lab)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

    model = ESM2DeepLoc(esm_dim=1280).to(device)
    optimizer = AdamW(model.parameters(), lr=1e-3)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    patience_counter = 0
    for epoch in range(10):
        # ✅ 수정2: criterion 제거 (4개 인자)
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss = validate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        if (epoch + 1) % 2 == 0:
            print(f"    Epoch {epoch+1}: Train {train_loss:.4f} | Val {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), f'{SAVE_PATH}deeploc_fold_{fold}_best.pth')
        else:
            patience_counter += 1
            if patience_counter >= 3:
                break

    fold_results.append(best_val_loss)
    print(f"  ✅ Fold {fold} 완료 | Best Val: {best_val_loss:.4f}")

print(f"\n🎉 DeepLoc 4-Fold CV 완료!")
print(f"평균 Val Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")


In [ ]:
test_loss = validate(best_model, test_loader, criterion, device)


In [ ]:
# Partition 4로 테스트
test_mask = (partitions == 4)
test_emb = embeddings[test_mask]
test_lab = labels[test_mask]
# 최고 모델로 평가


In [ ]:
# 4개 모델 앙상블
ensemble_pred = (pred0 + pred1 + pred2 + pred3) / 4


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Optimizer - AdamW
optimizer = AdamW(model.parameters(), lr=1e-3)

# Learning Rate Scheduler
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=4
)

print(f"✅ Optimizer와 Scheduler 정의 완료")
print(f"초기 학습률: 1e-3")
print(f"감소 인자: 0.5")
print(f"Patience: 4 에포크")


In [ ]:
def train_epoch(model, train_loader, optimizer, focal_loss, device):
    """
    1 에포크 훈련
    """
    model.train()
    total_loss = 0.0

    for batch_embeddings, batch_labels in train_loader:
        batch_embeddings = batch_embeddings.to(device)
        batch_labels = batch_labels.to(device)

        # Forward pass
        outputs = model(batch_embeddings)
        loss = focal_loss(outputs, batch_labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    return avg_loss


def validate(model, val_loader, focal_loss, device):
    """
    검증 데이터로 평가
    """
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch_embeddings, batch_labels in val_loader:
            batch_embeddings = batch_embeddings.to(device)
            batch_labels = batch_labels.to(device)

            outputs = model(batch_embeddings)
            loss = focal_loss(outputs, batch_labels)
            total_loss += loss.item()

    avg_loss = total_loss / len(val_loader)
    return avg_loss

print(f"✅ 훈련 및 검증 함수 정의 완료")


In [ ]:
from sklearn.model_selection import train_test_split

# Partition 0 데이터 다시 로드
p0_embeddings = np.load(f'{SAVE_PATH}embeddings_p0.npy')
p0_labels = np.load(f'{SAVE_PATH}labels_p0.npy')

# 훈련/검증 분할 (80% 훈련, 20% 검증)
train_embeddings, val_embeddings, train_labels, val_labels = train_test_split(
    p0_embeddings,
    p0_labels,
    test_size=0.2,
    random_state=42
)

# Dataset 만들기
train_dataset = ProteinDataset(train_embeddings, train_labels)
val_dataset = ProteinDataset(val_embeddings, val_labels)

# DataLoader 만들기
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"✅ DataLoader 생성 완료")
print(f"훈련 샘플: {len(train_dataset)}")
print(f"검증 샘플: {len(val_dataset)}")
print(f"훈련 배치 수: {len(train_loader)}")
print(f"검증 배치 수: {len(val_loader)}")


In [ ]:
# 확인 리스트
print("✅ 확인 사항:")
print(f"1. model: {type(model).__name__}")
print(f"2. optimizer: {type(optimizer).__name__}")
print(f"3. scheduler: {type(scheduler).__name__}")
print(f"4. focal_loss: {type(focal_loss).__name__}")
print(f"5. train_loader: {len(train_loader)} 배치")
print(f"6. val_loader: {len(val_loader)} 배치")


In [ ]:
# 설정
num_epochs = 10
early_stopping_patience = 5
best_val_loss = float('inf')
patience_counter = 0

# 훈련 히스토리 저장
history = {
    'train_loss': [],
    'val_loss': []
}

print("훈련 시작...")
print("-" * 50)

for epoch in range(num_epochs):
    # 훈련
    train_loss = train_epoch(model, train_loader, optimizer, focal_loss, device)

    # 검증
    val_loss = validate(model, val_loader, focal_loss, device)

    # 히스토리 저장
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    # Learning Rate Scheduler 업데이트
    scheduler.step(val_loss)

    # 출력
    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), f'{SAVE_PATH}best_model.pth')
    else:
        patience_counter += 1
        if patience_counter >= early_stopping_patience:
            print(f"\n🛑 Early Stopping (Patience: {early_stopping_patience})")
            break

print("-" * 50)
print("✅ 훈련 완료")


In [ ]:
# 최고 성능 모델 로드
model.load_state_dict(torch.load(f'{SAVE_PATH}best_model.pth'))

# 검증 데이터로 예측
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_embeddings, batch_labels in val_loader:
        batch_embeddings = batch_embeddings.to(device)
        outputs = model(batch_embeddings)

        # 0.5 이상이면 1, 아니면 0
        predictions = (outputs > 0.5).cpu().numpy()
        all_predictions.append(predictions)
        all_labels.append(batch_labels.numpy())

all_predictions = np.vstack(all_predictions)
all_labels = np.vstack(all_labels)

# 성능 지표
from sklearn.metrics import accuracy_score, f1_score, hamming_loss

accuracy = accuracy_score(all_labels, all_predictions)
hamming = hamming_loss(all_labels, all_predictions)

print(f"✅ 성능 평가")
print(f"정확도: {accuracy:.4f}")
print(f"해밍 손실: {hamming:.4f}")

# 각 클래스별 F1
class_names = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']
print(f"\n클래스별 F1 Score:")
for i, name in enumerate(class_names):
    f1 = f1_score(all_labels[:, i], all_predictions[:, i])
    print(f"  {name}: {f1:.4f}")


In [ ]:
# 라벨 분포 확인
print("훈련 데이터 라벨 분포:")
print(f"Peripheral: {all_labels[:, 0].sum():.0f}")
print(f"Transmembrane: {all_labels[:, 1].sum():.0f}")
print(f"LipidAnchor: {all_labels[:, 2].sum():.0f}")
print(f"Soluble: {all_labels[:, 3].sum():.0f}")

print(f"\n검증 데이터 샘플 수: {len(all_labels)}")

# 첫 번째 배치 확인
batch_emb, batch_lbl = next(iter(val_loader))
print(f"\n첫 배치 라벨 (처음 3개):")
print(batch_lbl[:3])

# 예측값 확인
print(f"\n모델 예측값 (처음 3개):")
with torch.no_grad():
    batch_emb = batch_emb.to(device)
    batch_pred = model(batch_emb)
    print(batch_pred[:3])


In [ ]:
# 최고 성능 모델 로드
model.load_state_dict(torch.load(f'{SAVE_PATH}best_model.pth'))

# 검증 데이터로 최종 평가
final_val_loss = validate(model, val_loader, focal_loss, device)

print(f"✅ 최고 성능 모델 로드 완료")
print(f"최종 검증 손실: {final_val_loss:.6f}")

# 훈련 히스토리 확인
print(f"\n훈련 히스토리:")
print(f"초기 훈련 손실: {history['train_loss'][0]:.6f}")
print(f"최종 훈련 손실: {history['train_loss'][-1]:.6f}")
print(f"초기 검증 손실: {history['val_loss'][0]:.6f}")
print(f"최종 검증 손실: {history['val_loss'][-1]:.6f}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, hamming_loss

# 검증 데이터로 예측
model.eval()
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_embeddings, batch_labels in val_loader:
        batch_embeddings = batch_embeddings.to(device)
        outputs = model(batch_embeddings)

        # 0.5 이상이면 1, 아니면 0
        predictions = (outputs > 0.5).cpu().numpy()
        all_predictions.append(predictions)
        all_labels.append(batch_labels.numpy())

all_predictions = np.vstack(all_predictions)
all_labels = np.vstack(all_labels)

# 성능 지표 계산
accuracy = accuracy_score(all_labels, all_predictions)
hamming = hamming_loss(all_labels, all_predictions)

print(f"✅ 성능 평가 완료")
print(f"\n검증 데이터 성능:")
print(f"정확도 (Accuracy): {accuracy:.4f}")
print(f"해밍 손실 (Hamming Loss): {hamming:.4f}")

# 각 클래스별 성능
class_names = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']
print(f"\n클래스별 F1 Score:")
for i, class_name in enumerate(class_names):
    f1 = f1_score(all_labels[:, i], all_predictions[:, i])
    print(f"  {class_name}: {f1:.4f}")


In [ ]:
# 라벨 분포 확인
print("검증 데이터 라벨 분포:")
print(f"Peripheral: {all_labels[:, 0].sum():.0f}")
print(f"Transmembrane: {all_labels[:, 1].sum():.0f}")
print(f"LipidAnchor: {all_labels[:, 2].sum():.0f}")
print(f"Soluble: {all_labels[:, 3].sum():.0f}")

print("\n예측된 라벨 분포:")
print(f"Peripheral: {all_predictions[:, 0].sum():.0f}")
print(f"Transmembrane: {all_predictions[:, 1].sum():.0f}")
print(f"LipidAnchor: {all_predictions[:, 2].sum():.0f}")
print(f"Soluble: {all_predictions[:, 3].sum():.0f}")

print(f"\n총 검증 샘플: {len(all_labels)}")

In [ ]:
# df_test의 클래스별 개수 확인
print("학습에 사용한 데이터 (df_test) 클래스별 개수:")
print("=" * 50)

class_names = ['Peripheral', 'Transmembrane', 'LipidAnchor', 'Soluble']

for class_name in class_names:
    count = test_df[class_name].sum()
    percentage = (count / len(test_df)) * 100
    print(f"{class_name}: {count:.0f} ({percentage:.1f}%)")

print("=" * 50)
print(f"총 샘플: {len(test_df)}")

# Partition별 분포도 확인
print(f"\nPartition별 분포:")
print(test_df['Partition'].value_counts().sort_index())
